In [63]:
import cv2
from ultralytics import YOLO

import easyocr

import time

import pandas as pd

# import numpy as np


In [64]:
# Initialisation du lecteur EasyOCR.
# liste de langues besoin (ex. ['en'] ou ['fr'] ou ['en','fr'])

reader = easyocr.Reader(['en', 'fr'], gpu=True)

In [65]:
model_name = "y12x_320px_15patience" 
score_board = "_score_board"

model_path = f"../../models/fine_tuned_models{score_board}/{model_name}/weights/best.pt"

# model_path = f"../../models/pretrainned_models/yolo12x.pt"

In [66]:
model = YOLO(model_path, task="detect")

In [67]:
source = '../../outputs/video_extract_img/video_test.mp4'

path_video_output = '../../outputs/video_annotees'

In [68]:
WINDOW_NAME = 'figther_tracking'
WINDOW_WIDTH = 1000   # Nouvelle largeur souhaitée
WINDOW_HEIGHT = 800  # Nouvelle hauteur souhaitée
WINDOW_POS_X = 1900   # Position horizontale sur l'écran
WINDOW_POS_Y = 100   # Position verticale sur l'écran

In [69]:
# Open the video file
video_path = source
cap = cv2.VideoCapture(video_path)

# Variable pour contrôler la fréquence de détection
last_detection_time = 0.0  # on stocke le temps de la dernière détection
DETECTION_INTERVAL = 1   # en secondes, ici 1 s

# Pour stocker l'image annotée la plus récente
annotated_frame = None
text0 ="_"
text1 ="_"
text2 ="_"
text3 ="_"
text4 ="_"
text5 ="_"
text6 = 0

recognized_text0 = ""
recognized_text1 = ""
recognized_text2 = ""
recognized_text3 = ""
recognized_text4 = ""
recognized_text5 = ""
recognized_text6 = ""

df_annotation = pd.DataFrame(columns=["fighter_1", "fighter_2", "chrono", "text3 ?", "text4 ?", "text5 ?", "mate" ])


# Loop through the video frames
while cap.isOpened():
    # Read a frame from the video
    success, frame = cap.read()
    
    if not success:
        break

        # Run  tracking on the frame, persisting tracks between frames
    results = model(
                                source=frame,
                                conf=0.75, 
                                iou=0.05, 
                                # classes=1, 
                                max_det=1
                            )
    
    # Récupère le temps actuel
    current_time = time.time()

    if current_time - last_detection_time >= DETECTION_INTERVAL:
        # Met à jour le temps de la dernière détection
        last_detection_time = current_time    


        # Récupère les infos de détection
        boxes = results[0].boxes
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

            # marge autour de la boîte englobante
            height, width, _ = frame.shape

            # margin = 25
            # x1 = max(0, x1 - margin)
            # y1 = max(0, y1 - margin)
            # x2 = min(width,  x2 + margin)
            # y2 = min(height, y2 + margin)

            x1 = 0
            y1 = 850
            x2 = 700
            y2 = 1200

            cropped = frame[y1:y2, x1:x2]



            # ----- Prétraitement pour EasyOCR -----

            # Convertir en niveaux de gris (si vous le souhaitez)
            gray = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)

            # Binarisation de base
            _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)
            # Optionnel : morphologie
            kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1,1))
            thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

            # Redimensionner si trop petit
            h_cropped, w_cropped = thresh.shape
            if w_cropped < 100 or h_cropped < 40:
                scale = 2
                thresh = cv2.resize(thresh, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)

            # EasyOCR attend en principe une image en RGB (3 canaux)
            thresh_rgb = cv2.cvtColor(thresh, cv2.COLOR_GRAY2RGB)

            # ----- Lecture OCR avec EasyOCR -----
            results_ocr = reader.readtext(
                                                                        thresh_rgb, 
                                                                        detail=1,
                                                                        allowlist=':ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789',
                                                                        decoder='greedy',
                                                                        paragraph=False,
                                                                        min_size=10,
                                                                        text_threshold=0.7,
                                                                        low_text=0.6,
                                                                        # link_threshold=0.4,
                                                                        # contrast_ths=0.05,
                                                                        # adjust_contrast=0.7,
                                                                        # filter_ths=0.003,
                                                                        # y_ths=0.5,
                                                                        # width_ths=0.5,
                                                                        # height_ths=0.5,
                                                                        # add_margin=0.1,
                                                                        # rotation_info=[0, 90, 180, 270]
                                                                        )

            # results_ocr est une liste de ( [ [x0,y0], [x1,y1], [x2,y2], [x3,y3] ], texte, confiance )

            # On va simplement concaténer ou afficher le premier champ texte
            if results_ocr :
                if len(results_ocr) > 0:
                    # Si on ne s'intéresse qu'au premier bloc détecté
                    coords0, recognized_text0, confidence0 = results_ocr[0]
                    text0 = f"fighter_1 :  {recognized_text0.strip()} confiance : {confidence0:.2f}"
                    print(f"fighter_1 : {text0}, confiance : {confidence0:.2f}")

                if len(results_ocr) > 1:
                    # Si on ne s'intéresse qu'au premier bloc détecté
                    coords1, recognized_text1, confidence1 = results_ocr[1]
                    text1 = f"fighter_2 :  {recognized_text1.strip()} confiance : {confidence1:.2f}"
                    print(f"fighter_2 : {text1}, confiance : {confidence1:.2f}")

                if len(results_ocr) > 2:
                    # Si on ne s'intéresse qu'au premier bloc détecté
                    coords2, recognized_text2, confidence2 = results_ocr[2]
                    text2 = f"chrono :  {recognized_text2.strip()} confiance : {confidence2:.2f}"
                    print(f"Chrono : {text2}, confiance : {confidence2:.2f}")

                if len(results_ocr) > 3:
                    # Si on ne s'intéresse qu'au premier bloc détecté
                    coords3, recognized_text3, confidence3 = results_ocr[3]
                    text3 = f"text ? :  {recognized_text3.strip()} confiance : {confidence3:.2f}"
                    print(f"text ? : {text3}, confiance : {confidence3:.2f}")

                if len(results_ocr) > 4:
                    # Si on ne s'intéresse qu'au premier bloc détecté
                    coords4, recognized_text4, confidence4 = results_ocr[4]
                    text4 = f"text ? :  {recognized_text4.strip()} confiance : {confidence4:.2f}"
                    print(f"text ? : {text4}, confiance : {confidence4:.2f}")

                if len(results_ocr) > 5:
                    # Si on ne s'intéresse qu'au premier bloc détecté
                    coords5, recognized_text5, confidence5 = results_ocr[5]
                    text5 = f"text ? :  {recognized_text5.strip()} confiance : {confidence5:.2f}"
                    print(f"text ? : {text5}, confiance : {confidence5:.2f}")

                text6 = f"nombre de detection : {len(results_ocr)}"

                nouvelle_ligne = {
                                    "fighter_1": recognized_text0.strip(),
                                    "fighter_2": recognized_text1.strip(),
                                    "chrono": recognized_text2.strip(),
                                    "text3 ?": recognized_text3.strip(),
                                    "text4 ?": recognized_text4.strip(),
                                    "text5 ?": recognized_text5.strip(),
                                    "mate" : 0
                                }

                if df_annotation.empty:
                    df_annotation.loc[len(df_annotation)] = nouvelle_ligne

                else:
                    dict_info_previous_line = df_annotation.iloc[-1].to_dict()

                    if dict_info_previous_line != nouvelle_ligne:

                        df_annotation.loc[len(df_annotation)] = nouvelle_ligne
                    
                    
                    if  dict_info_previous_line["chrono"] == nouvelle_ligne["chrono"]:
                        df_annotation.loc[df_annotation.index[0], "mate"] = 1



                    

            else:
                text = ""
                print("Aucun texte détecté")

        # ----- Fin du bloc de détection -----

    else:
        # Si on n'effectue pas de détection sur cette frame,
        # on peut simplement copier la frame « brute » pour l'affichage
        # ou réutiliser la dernière image annotée si vous préférez.
        if annotated_frame is None:
            annotated_frame = frame.copy()
        else:
            # Option 1 : réutiliser la dernière image annotée
            # annotated_frame = annotated_frame

            # Option 2 : ou bien afficher la frame en direct (non annotée)
            annotated_frame = frame.copy()

    # Annoter la frame (détection YOLO)
    annotated_frame =  results[0].plot()


    # Affichage du texte sur l'image
    cv2.putText(
        annotated_frame,
        text0,         # Le texte à afficher
        (150, 300),               # Coordonnées (x, y) dans l'image
        cv2.FONT_HERSHEY_SIMPLEX, # Police 
        1,                        # Taille du texte (scale)
        (0, 255, 0),              # Couleur (B, G, R)  vert
        2                         # Épaisseur du trait
    )

    cv2.putText(
        annotated_frame,
        text1,         # Le texte à afficher
        (150, 340),               # Coordonnées (x, y) dans l'image
        cv2.FONT_HERSHEY_SIMPLEX, # Police 
        1,                        # Taille du texte (scale)
        (0, 255, 0),              # Couleur (B, G, R)  vert
        2                         # Épaisseur du trait
    )   

    cv2.putText(
        annotated_frame,
        text2,         # Le texte à afficher
        (150, 380),               # Coordonnées (x, y) dans l'image
        cv2.FONT_HERSHEY_SIMPLEX, # Police 
        1,                        # Taille du texte (scale)
        (0, 255, 0),              # Couleur (B, G, R)  vert
        2                         # Épaisseur du trait
    ) 

    cv2.putText(
        annotated_frame,
        text3,         # Le texte à afficher
        (150, 420),               # Coordonnées (x, y) dans l'image
        cv2.FONT_HERSHEY_SIMPLEX, # Police 
        1,                        # Taille du texte (scale)
        (0, 255, 0),              # Couleur (B, G, R)  vert
        2                         # Épaisseur du trait
    )

    cv2.putText(
        annotated_frame,
        text4,         # Le texte à afficher
        (150, 460),               # Coordonnées (x, y) dans l'image
        cv2.FONT_HERSHEY_SIMPLEX, # Police 
        1,                        # Taille du texte (scale)
        (0, 255, 0),              # Couleur (B, G, R)  vert
        2                         # Épaisseur du trait
    )

    cv2.putText(
        annotated_frame,
        text5,         # Le texte à afficher
        (150, 500),               # Coordonnées (x, y) dans l'image
        cv2.FONT_HERSHEY_SIMPLEX, # Police 
        1,                        # Taille du texte (scale)
        (0, 255, 0),              # Couleur (B, G, R)  vert
        2                         # Épaisseur du trait
    )

    # Affichage du texte sur l'image
    cv2.putText(
        annotated_frame,
        "x1 y1",         # Le texte à afficher
        (x1, y1),                 # Coordonnées (x, y) dans l'image
        cv2.FONT_HERSHEY_SIMPLEX, # Police 
        1,                        # Taille du texte (scale)
        (0, 255, 0),              # Couleur (B, G, R)  vert
        2                         # Épaisseur du trait
    )

        # Affichage du texte sur l'image
    cv2.putText(
        annotated_frame,
        "x2, y2",         # Le texte à afficher
        (x2, y2),                 # Coordonnées (x, y) dans l'image
        cv2.FONT_HERSHEY_SIMPLEX, # Police 
        1,                        # Taille du texte (scale)
        (0, 255, 0),              # Couleur (B, G, R) vert
        2                         # Épaisseur du trait
    )

    # Crée et configure la fenêtre
    cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(WINDOW_NAME, WINDOW_WIDTH, WINDOW_HEIGHT)
    cv2.moveWindow(WINDOW_NAME, WINDOW_POS_X, WINDOW_POS_Y)

    # Affiche le résultat (soit la frame annotée, soit la frame brute)
    cv2.imshow(WINDOW_NAME, annotated_frame)

    # Quitte si on appuie sur "q"
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# Release the video capture object and close the display window
cap.release()
cv2.destroyAllWindows()



0: 192x320 1 incrus_score_board, 30.7ms
Speed: 1.1ms preprocess, 30.7ms inference, 1.9ms postprocess per image at shape (1, 3, 192, 320)
fighter_1 : fighter_1 :  2 confiance : 0.24, confiance : 0.24

0: 192x320 1 incrus_score_board, 29.0ms
Speed: 1.1ms preprocess, 29.0ms inference, 1.8ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 1 incrus_score_board, 29.6ms
Speed: 1.1ms preprocess, 29.6ms inference, 1.5ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 1 incrus_score_board, 29.3ms
Speed: 0.9ms preprocess, 29.3ms inference, 2.4ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 1 incrus_score_board, 29.1ms
Speed: 1.1ms preprocess, 29.1ms inference, 1.7ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 1 incrus_score_board, 30.5ms
Speed: 0.9ms preprocess, 30.5ms inference, 1.9ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 1 incrus_score_board, 46.7ms
Speed: 0.9ms preprocess, 46.7ms inference, 3.4ms postprocess per imag

In [70]:
df_annotation.to_csv(f"{path_video_output}/df_annotation.csv", index=False)
print("DataFrame saved to df_annotation.csv")

DataFrame saved to df_annotation.csv


In [71]:
df_annotation


,fighter_1,fighter_2,chrono,text3 ?,text4 ?,text5 ?,mate
0,2,,,,,,1
1,7,,,,,,0
2,42,,,,,,0
3,2,,,,,,0
4,,,,MIAIRUYAMA,,,0
5,JP,,,MIAIRUYAMA,,,0
6,JIP,,,MIAIRUYAMA,,,0
7,JP,,,MIAIRUYAMA,,,0
8,P,,,MIAIRUYAMA,,,0
9,JPN,,,MIAIRUYAMA,,,0


In [74]:

df_annotation['text4 ?'].value_counts()

text4 ?
            16
1            4
MARUYANA     3
L            1
MARUMANA     1
MARUVANA     1
MAHUAMA      1
MARUANA      1
MARUYAMA     1
Name: count, dtype: int64